# 08 — Second-round reviewer analyses (CPU; roughly 1–3 hours, estimate)
All **post-registration and exploratory**. Run after notebook 07; needs `results/reviewer/reviewer_analyses.json`.

| Section | Review item | Cost |
|---|---|---|
| S1 | Q1: registered estimands without the six pilot datasets | seconds |
| S2 | concern 3, Q2: all hybrid rules tried, gate width, paired tests vs competence | minutes |
| S3 | concern 4, Q2: **fully nested selection** (probes refitted inside each outer split), pools with n ≤ 12.5k | most of the time |
| S4 | concern 9: probe C ∈ {0.1, 1, 10} on DTD, Aircraft, MHIST | ~10–20 min |
| S5 | Q4: per-pool consistency of the weak association, leave-one-pool-out | seconds |
| S6 | concern 2: Benjamini–Hochberg and Holm adjustment of the exploratory p-values | seconds |
| S7 | minor: signed gap and fold interval of every primary pair | seconds |
| S8 | concern 9: Lunit and Kaiko ports vs the official releases; DINOv2 at 224 px vs the official model | downloads ~2 GB |

A GPU runtime makes S3 and S4 much faster, but they run on CPU. Each section is saved as soon as it finishes, so
a disconnect loses at most the section in progress.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
assert os.path.exists(os.path.join(CODE, "config.py")), "run 00_setup_and_prereg.ipynb first"
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import importlib, config as CFG
importlib.reload(CFG)
print("Drive:", CFG.DRIVE, "| project:", CFG.PROJECT)

In [ ]:
os.makedirs(os.path.join(PROJECT, "code"), exist_ok=True)
print("ready")

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/reviewer2.py
"""Second-round exploratory analyses (all POST-REGISTRATION).

  S1 pilot_overlap_exclusion  Q1  registered estimands without the six pilot datasets
  S2 gated_rule               concern 3 / Q2  all rules tried; gated vs competence (paired);
                                   gate width 0.5 / 1 / 1.5 / 3 points
  S3 nested_regret            concern 4 / Q2  fully nested selection: probes for the selection
                                   statistics are refitted inside each outer training split
  S4 c_sensitivity            concern 9   probe C in {0.1, 1, 10} on the smallest pools
  S5 association_consistency  Q4  per-pool slopes and leave-one-pool-out for the weak association
  S6 multiplicity             concern 2   Benjamini-Hochberg and Holm over the exploratory p-values
  S7 primary_pairs            minor       signed gap and fold interval of every primary pair
  S8 encoder_validation       concern 9   ported weights vs official releases (Lunit, Kaiko),
                                   DINOv2 at 224 px vs the official model

Reads the same saved artefacts as reviewer.py; S3, S4 and S8 additionally refit
linear probes or load weights, on CPU (GPU if present). Results carry the same
EXPLORATORY label.
"""

from __future__ import annotations

import json
import os
import time
import warnings

import numpy as np
from scipy import stats

import config as CFG
import folds as FO
import fusion
import reviewer as RV
from inference import cluster_robust_ols, tost_equivalence

EXPLORATORY = RV.EXPLORATORY
PILOT_DATASETS = ("bracs", "sicapv2", "mhist", "dtd", "aircraft", "eurosat")


def _paired(d):
    d = np.asarray(d, float)
    out = {"n": int(len(d)), "mean": float(d.mean()), "better_in": int((d > 1e-9).sum()),
           "worse_in": int((d < -1e-9).sum()), "tie": int((np.abs(d) <= 1e-9).sum())}
    if len(d) > 1 and np.any(d != 0):
        t = stats.ttest_1samp(d, 0.0)
        tc = stats.t.ppf(0.975, len(d) - 1)
        se = d.std(ddof=1) / np.sqrt(len(d))
        out.update(p_t=float(t.pvalue), ci_low=float(d.mean() - tc * se), ci_high=float(d.mean() + tc * se))
        try:
            out["p_wilcoxon"] = float(stats.wilcoxon(d).pvalue)
        except ValueError:
            pass
    return out


# ---------------------------------------------------------------- S1
def pilot_overlap_exclusion(recs):
    keep = [r for r in recs if r["pool"] not in PILOT_DATASETS]
    out = {"note": EXPLORATORY, "excluded": list(PILOT_DATASETS), "pools": [r["pool"] for r in keep]}
    for s in RV.SUBSETS:
        R = RV._subset(keep, s)
        P = RV._pairs(R)
        prim = RV._primary(P)
        regret = {}
        for r in R:
            c = r["candidates"]
            best = max(x["utility"] for x in c)
            for name, key, rev in (("lowest_cka", "cka", False), ("competence", "indiv_acc", True),
                                   ("unique", "headroom_contrib", True)):
                pick = (max if rev else min)(c, key=lambda x: x[key])
                regret.setdefault(name, []).append(best - pick["utility"])
        d = [p["diff"] for p in P]; cl = [p["pool"] for p in P]
        out[s] = {"primary": RV._cr([p["diff"] for p in prim], [p["pool"] for p in prim]),
                  "all_pairs": RV._cr(d, cl), "tost_all_pairs": RV._tost(d, cl),
                  "regret": {k: float(np.mean(v)) for k, v in regret.items()},
                  "pools": len(R)}
    return out


# ---------------------------------------------------------------- S2
def gated_rule(recs, rv_json, results_dir=None, gates=(0.5, 1.0, 1.5, 3.0)):
    """Cross-fitted regret of the gated rule for several gate widths, and paired
    comparisons with competence. Rules tried in the first exploratory round: the
    four registered rules, the gated rule (gate 1.5) and the rank-sum rule; that
    is the complete list, all fixed in code before that round was run."""
    rows = rv_json["R5_regret_crossfit"]["per_pool"]
    out = {"note": EXPLORATORY,
           "rules_tried_round1": ["lowest_cka", "competence", "unique", "disagreement",
                                  "gated_diversity (gate 1.5)", "rank_sum"],
           "paired_vs_competence_crossfit": {}}
    for s in RV.SUBSETS:
        R = [x for x in rows if x["domain"] in RV.SUBSETS[s]]
        if len(R) < 2:
            continue
        out["paired_vs_competence_crossfit"][s] = {
            rule: _paired([x["competence_regret_crossfit"] - x[f"{rule}_regret_crossfit"] for x in R])
            for rule in ("gated_diversity", "rank_sum", "unique")}
    # gate width sensitivity (cross-fitted, computed afresh)
    per = []
    for r in recs:
        Z, y, masks = RV.load_logits(r, results_dir)
        cands = [c["name"] for c in r["candidates"]]
        U = {c: np.array(r["utility_per_fold"]["mean_logits"][c]) for c in cands}
        oracle = max(U[c].mean() for c in cands)
        cka = {c: r["cka_with_anchor"][c] for c in cands}
        pred = {c: Z[c].argmax(1) for c in cands}
        row = {"pool": r["pool"], "domain": r["domain"]}
        for g in gates:
            got = []
            for f, m in enumerate(masks):
                acc = {c: float((pred[c][~m] == y[~m]).mean() * 100) for c in cands}
                top = max(acc.values())
                ok = [c for c in cands if acc[c] >= top - g]
                got.append(U[min(ok, key=lambda c: (cka[c], -acc[c]))][f])
            row[f"gate_{g}"] = float(oracle - np.mean(got))
        per.append(row)
    out["gate_width_crossfit"] = {}
    for s in RV.SUBSETS:
        R = [x for x in per if x["domain"] in RV.SUBSETS[s]]
        if R:
            out["gate_width_crossfit"][s] = {str(g): float(np.mean([x[f"gate_{g}"] for x in R])) for g in gates}
    # consistency check: gate 1.5 must reproduce round 1
    r1 = {x["pool"]: x["gated_diversity_regret_crossfit"] for x in rows}
    worst = max(abs(x["gate_1.5"] - r1[x["pool"]]) for x in per if x["pool"] in r1)
    assert worst < 1e-9, f"gate 1.5 does not reproduce round-1 gated regret ({worst})"
    out["per_pool_gate_width"] = per
    return out


# ---------------------------------------------------------------- S3
def _load_features(ds, names):
    import extract as X_
    F = {}
    for e in names:
        p = X_._paths(ds, e)[1]
        if not os.path.exists(p):
            return None
        F[e] = np.load(p)
    return F


def nested_regret(recs, results_dir=None, n_max=12500, inner_k=4, gates=(1.5,), verbose=True,
                  backend="torch"):
    """Fully nested selection. For outer fold f, every label-using selection
    statistic is computed from INNER held-out logits: probes refitted by inner
    k-fold CV on the outer training rows only, so no label of fold f is used
    anywhere in selection. The chosen candidate is scored by its registered
    utility on fold f. Pools with more than n_max inputs are skipped (cost)."""
    per = []
    for r in recs:
        if r["n_inputs"] > n_max:
            continue
        t0 = time.time()
        man = json.load(open(os.path.join(CFG.MANIFEST_DIR, f"{r['pool']}.json")))
        Z, y, masks = RV.load_logits(r, results_dir)
        base = list(r["base"])
        cands = [c["name"] for c in r["candidates"]]
        F = _load_features(r["pool"], cands + base)
        if F is None:
            if verbose:
                print(f"  [{r['pool']}] features missing; skipped")
            continue
        groups = np.asarray(man["groups"]) if (man.get("groups") and r["grouped_folds"]) else None
        U = {c: np.array(r["utility_per_fold"]["mean_logits"][c]) for c in cands}
        oracle = max(U[c].mean() for c in cands)
        cka = {c: r["cka_with_anchor"][c] for c in cands}
        picks = {k: [] for k in ("lowest_cka", "competence", "unique", "disagreement", "rank_sum",
                                 *[f"gated_{g}" for g in gates])}
        for f, m in enumerate(masks):
            R_ = np.where(~m)[0]
            yR = y[R_]
            gR = None if groups is None else groups[R_]
            im = FO.make_folds(yR, k=inner_k, groups=gR, seed=CFG.SEED + 100 + f)
            Zi = {e: FO.per_fold_logits(F[e][R_], yR, im, backend=backend) for e in cands + base}
            pred = {e: Zi[e].argmax(1) for e in Zi}
            bp = fusion.fuse([Zi[b] for b in base], "mean_logits").argmax(1)
            base_any = np.zeros(len(R_), bool)
            for b in base:
                base_any |= pred[b] == yR
            acc = {c: float((pred[c] == yR).mean() * 100) for c in cands}
            uniq = {c: float(((pred[c] == yR) & ~base_any).mean() * 100) for c in cands}
            dis = {c: float((pred[c] != bp).mean()) for c in cands}
            for k in picks:
                if k.startswith("gated_"):
                    g = float(k.split("_")[1]); top = max(acc.values())
                    ok = [c for c in cands if acc[c] >= top - g]
                    sel = min(ok, key=lambda c: (cka[c], -acc[c]))
                else:
                    sel = RV._select(k, cands, acc, cka, uniq, dis)
                picks[k].append(U[sel][f])
        row = {"pool": r["pool"], "domain": r["domain"], "n": r["n_inputs"],
               **{f"{k}_regret_nested": float(oracle - np.mean(v)) for k, v in picks.items()},
               "minutes": round((time.time() - t0) / 60, 1)}
        per.append(row)
        if verbose:
            print(f"  [{r['pool']}] nested selection done in {row['minutes']} min", flush=True)
    out = {"note": EXPLORATORY, "inner_k": inner_k, "n_max": n_max, "per_pool": per, "summary": {}}
    for s in RV.SUBSETS:
        R = [x for x in per if x["domain"] in RV.SUBSETS[s]]
        if not R:
            continue
        summ = {"pools": len(R)}
        for k in ("lowest_cka", "competence", "unique", "disagreement", "rank_sum", *[f"gated_{g}" for g in gates]):
            summ[k] = float(np.mean([x[f"{k}_regret_nested"] for x in R]))
        summ["gated_vs_competence"] = _paired([x["competence_regret_nested"] - x[f"gated_{gates[0]}_regret_nested"] for x in R])
        summ["competence_vs_lowest_cka"] = _paired([x["lowest_cka_regret_nested"] - x["competence_regret_nested"] for x in R])
        out["summary"][s] = summ
    return out


# ---------------------------------------------------------------- S4
def c_sensitivity(recs, pools=("dtd", "aircraft", "mhist"), Cs=(0.1, 1.0, 10.0), results_dir=None,
                  backend="torch", verbose=True):
    """Refit the per-fold probes with other C values on the registered folds and
    recompute every candidate's utility, the matched pairs' D (same pairs) and the
    regrets of the registered rules. C = 1 must reproduce the registered values."""
    out = {"note": EXPLORATORY, "pools": {}}
    for r in recs:
        if r["pool"] not in pools:
            continue
        Z, y, masks = RV.load_logits(r, results_dir)
        names = sorted(Z)
        F = _load_features(r["pool"], names)
        if F is None:
            continue
        base = list(r["base"]); cands = [c["name"] for c in r["candidates"]]
        res = {}
        for C in Cs:
            t0 = time.time()
            Zc = Z if C == 1.0 else {e: _logits_C(F[e], y, masks, C, backend) for e in names}
            acc = {e: float((Zc[e].argmax(1) == y).mean() * 100) for e in names}
            def util(c):
                v = []
                for m in masks:
                    ab = (fusion.fuse([Zc[b][m] for b in base], "mean_logits").argmax(1) == y[m]).mean()
                    ac = (fusion.fuse([Zc[b][m] for b in base + [c]], "mean_logits").argmax(1) == y[m]).mean()
                    v.append((ac - ab) * 100)
                return float(np.mean(v))
            u = {c: util(c) for c in cands}
            if C == 1.0:
                reg = {c["name"]: c["utility"] for c in r["candidates"]}
                assert max(abs(u[c] - reg[c]) for c in cands) < 1e-9, "C=1 must reproduce registered utilities"
            D = [u[p["high"]] - u[p["low"]] for p in r["pairs"] if p["category"] == "matched"]
            best = max(u.values())
            cka = {c: r["cka_with_anchor"][c] for c in cands}
            res[str(C)] = {"mean_indiv_acc": float(np.mean([acc[c] for c in cands])),
                           "matched_D_mean": float(np.mean(D)) if D else None, "matched_pairs": len(D),
                           "regret_lowest_cka": best - u[min(cands, key=lambda c: cka[c])],
                           "regret_competence": best - u[max(cands, key=lambda c: acc[c])],
                           "minutes": round((time.time() - t0) / 60, 1)}
            if verbose:
                print(f"  [{r['pool']}] C={C}: done in {res[str(C)]['minutes']} min", flush=True)
        out["pools"][r["pool"]] = res
    return out


def _logits_C(X, y, masks, C, backend):
    import folds as F_
    return F_.per_fold_logits(X, y, list(masks), C=C, backend=backend)


# ---------------------------------------------------------------- S5
def association_consistency(rv_json, measures=("linear_cka_registered", "linear_cka_base", "mutual_knn_anchor",
                                               "rbf_cka_anchor")):
    rows = rv_json["R7_similarity"]["candidates"]
    pools = sorted({x["pool"] for x in rows})
    out = {"note": EXPLORATORY, "per_pool_sign": {}, "leave_one_pool_out": {}}
    for meas in measures:
        signs, slopes = [], {}
        for p in pools:
            R = [x for x in rows if x["pool"] == p]
            if len(R) < 4:
                continue
            X = np.column_stack([np.ones(len(R)), [x[meas] for x in R], [x["indiv_acc"] for x in R]])
            b = np.linalg.lstsq(X, np.array([x["utility"] for x in R]), rcond=None)[0]
            slopes[p] = float(b[1]); signs.append(np.sign(b[1]))
        loo = []
        for drop in pools:
            R = [x for x in rows if x["pool"] != drop]
            cl = np.array([x["pool"] for x in R]); u = np.array([x["utility"] for x in R])
            v = np.array([x[meas] for x in R]); a = np.array([x["indiv_acc"] for x in R])
            cen = lambda z: z - np.array([z[cl == q].mean() for q in cl])
            reg = cluster_robust_ols(cen(u), cl, x=np.column_stack([cen(v), cen(a)]))
            loo.append({"dropped": drop, "coef": float(reg["estimate"][1]), "p": float(reg["p"][1])})
        out["per_pool_sign"][meas] = {"negative_in": int(sum(s < 0 for s in signs)),
                                      "positive_in": int(sum(s > 0 for s in signs)), "pools": len(signs),
                                      "per_pool_slope": slopes}
        out["leave_one_pool_out"][meas] = {"coef_range": [min(l["coef"] for l in loo), max(l["coef"] for l in loo)],
                                           "p_range": [min(l["p"] for l in loo), max(l["p"] for l in loo)],
                                           "most_influential": max(loo, key=lambda l: l["coef"])["dropped"],
                                           "rows": loo}
    return out


# ---------------------------------------------------------------- S6
def _bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); prev = 1.0
    for rank, i in reversed(list(enumerate(o, 1))):
        prev = min(prev, p[i] * n / rank); q[i] = prev
    return q


def _holm(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    adj = np.empty(n); run = 0.0
    for rank, i in enumerate(o):
        run = max(run, min(1.0, p[i] * (n - rank))); adj[i] = run
    return adj


def multiplicity(rv_json):
    """Families: (a) Table 4: the seven similarity measures over all pools;
    (b) every candidate-level coefficient p-value of R7 (7 measures x 5 subsets);
    (c) every exploratory interval-based p-value reported for D in R1-R4, R9."""
    out = {"note": EXPLORATORY + "; adjusted p-values are descriptive"}
    r7 = rv_json["R7_similarity"]
    fam_a = [(m, r7["all"][m]["p"]) for m in RV.MEASURES]
    fam_b = [(f"{s}/{m}", r7[s][m]["p"]) for s in RV.SUBSETS if s in r7 for m in RV.MEASURES]
    fam_c = []
    for s in RV.SUBSETS:
        a = (rv_json["R1_competence_gap"].get(s) or {}).get("D_adjusted_all_pairs")
        if a: fam_c.append((f"R1 gap-adjusted D/{s}", a["p"]))
        for c, v in (rv_json["R2_caliper"].get(s) or {}).items():
            if v and v.get("all_pairs"): fam_c.append((f"R2 caliper {c}/{s}", v["all_pairs"]["p"]))
        v = rv_json["R3_strong"].get(s)
        if v and v.get("all_pairs"): fam_c.append((f"R3 strong/{s}", v["all_pairs"]["p"]))
        v = rv_json["R4_dependence"].get(s)
        if v and v["two_way_pool_x_encoder_pair"].get("p") is not None:
            fam_c.append((f"R4 two-way/{s}", v["two_way_pool_x_encoder_pair"]["p"]))
    for name, fam in (("table4_all_pools", fam_a), ("R7_all_subsets", fam_b), ("D_robustness", fam_c)):
        fam = [(k, p) for k, p in fam if p is not None and np.isfinite(p)]
        ps = [p for _, p in fam]
        q, h = _bh(ps), _holm(ps)
        out[name] = [{"test": k, "p": float(p), "q_bh": float(qq), "p_holm": float(hh)}
                     for (k, p), qq, hh in zip(fam, q, h)]
        out[name + "_min_q"] = float(min(q)) if len(q) else None
    return out


# ---------------------------------------------------------------- S7
def primary_pairs(recs):
    rows = []
    for r in recs:
        for p in r["pairs"]:
            if p.get("primary"):
                fd = np.array(p["fold_diffs"]); tc = stats.t.ppf(0.975, len(fd) - 1)
                se = fd.std(ddof=1) / np.sqrt(len(fd))
                rows.append({"pool": r["pool"], "high": p["high"], "low": p["low"], "cka_gap": p["cka_gap"],
                             "signed_gap": p["acc_high"] - p["acc_low"], "D": p["diff"],
                             "fold_ci_low": float(fd.mean() - tc * se), "fold_ci_high": float(fd.mean() + tc * se)})
    return rows


# ---------------------------------------------------------------- S8
def _state(model):
    return {k: v.detach().float().cpu() for k, v in model.state_dict().items()}


def _compare_states(a, b):
    common = sorted(set(a) & set(b))
    same_shape = [k for k in common if a[k].shape == b[k].shape]
    diffs = [float((a[k] - b[k]).abs().max()) for k in same_shape]
    return {"tensors_a": len(a), "tensors_b": len(b), "common": len(common), "same_shape": len(same_shape),
            "max_abs_diff": max(diffs) if diffs else None,
            "only_in_a": sorted(set(a) - set(b))[:6], "only_in_b": sorted(set(b) - set(a))[:6]}


def _dinov2_official(name):
    """Meta's DINOv2 model code and weights. Tries torch.hub; if GitHub's API refuses
    (rate limit), downloads the repository as an archive and loads it locally."""
    import torch
    try:
        return torch.hub.load("facebookresearch/dinov2", name, trust_repo=True)
    except Exception:
        import io, urllib.request, zipfile
        root = os.path.join(os.environ.get("TORCH_HOME", "/tmp"), "dinov2_src")
        src = os.path.join(root, "dinov2-main")
        if not os.path.isdir(src):
            data = urllib.request.urlopen(
                "https://codeload.github.com/facebookresearch/dinov2/zip/refs/heads/main", timeout=120).read()
            zipfile.ZipFile(io.BytesIO(data)).extractall(root)
        return torch.hub.load(src, name, source="local", trust_repo=True)


def encoder_validation(images=None, device="cpu", verbose=True):
    """Ported weights vs official releases, tensor by tensor; and the DINOv2
    encoders as we run them (224 px, resampled position embedding) vs the official
    model on the same images (CKA of the class token)."""
    import torch
    import timm
    from cka import linear_cka
    import encoders as E
    out = {"note": "validation of encoder loading; not an analysis of the study"}
    # Lunit DINO ViT-S/16: official release vs the timm port we use
    try:
        url = ("https://github.com/lunit-io/benchmark-ssl-pathology/releases/download/"
               "pretrained-weights/dino_vit_small_patch16_ep200.torch")
        sd = torch.hub.load_state_dict_from_url(url, map_location="cpu", progress=False)
        sd = sd.get("state_dict", sd.get("model", sd)) if isinstance(sd, dict) else sd
        sd = {k.replace("module.", "").replace("backbone.", ""): v.float() for k, v in sd.items()}
        port = timm.create_model("hf-hub:1aurent/vit_small_patch16_224.lunit_dino", pretrained=True, num_classes=0)
        out["lunit_vits16"] = {"source": url, **_compare_states(sd, _state(port))}
    except Exception as e:
        out["lunit_vits16"] = {"error": f"{type(e).__name__}: {e}"}
    if verbose: print("  lunit:", out["lunit_vits16"].get("max_abs_diff", out["lunit_vits16"].get("error")), flush=True)
    # Kaiko ViT-B/16: official torch.hub entry vs the timm port we use
    try:
        # the official release file named by kaiko's own hubconf.py (RELEASE_TAG 0.0.1); loading the
        # file directly avoids torch.hub's GitHub API calls, which are rate-limited
        url = "https://github.com/kaiko-ai/towards_large_pathology_fms/releases/download/0.0.1/vitb16.pth"
        sd = torch.hub.load_state_dict_from_url(url, map_location="cpu", progress=False)
        sd = sd.get("state_dict", sd.get("model", sd)) if isinstance(sd, dict) else sd
        sd = {k.replace("module.", ""): v.float() for k, v in sd.items() if hasattr(v, "float")}
        port = timm.create_model("hf-hub:1aurent/vit_base_patch16_224.kaiko_ai_towards_large_pathology_fms",
                                 pretrained=True, num_classes=0, dynamic_img_size=True)
        out["kaiko_vitb16"] = {"source": url, **_compare_states(sd, _state(port))}
    except Exception as e:
        out["kaiko_vitb16"] = {"error": f"{type(e).__name__}: {e}"}
    if verbose: print("  kaiko:", out["kaiko_vitb16"].get("max_abs_diff", out["kaiko_vitb16"].get("error")), flush=True)
    # DINOv2 weights: Meta's official release files vs timm's checkpoint at its native 518 px, tensor by
    # tensor (no GitHub API involved). The only change our 224-px run makes on top of these weights is
    # timm's bicubic resampling of the position embedding, checked on images below when the hub is reachable.
    for ours, tag, tid in (("dinov2_b", "vitb14", "vit_base_patch14_dinov2.lvd142m"),
                           ("dinov2_l", "vitl14", "vit_large_patch14_dinov2.lvd142m")):
        try:
            url = f"https://dl.fbaipublicfiles.com/dinov2/dinov2_{tag}/dinov2_{tag}_pretrain.pth"
            sd = torch.hub.load_state_dict_from_url(url, map_location="cpu", progress=False)
            sd = {k: v.float() for k, v in sd.items() if k != "mask_token"}
            native = timm.create_model(tid, pretrained=True, num_classes=0)
            out[f"{ours}_weights_vs_official"] = {"source": url, **_compare_states(sd, _state(native))}
            del native
        except Exception as e:
            out[f"{ours}_weights_vs_official"] = {"error": f"{type(e).__name__}: {e}"}
        if verbose:
            x = out[f"{ours}_weights_vs_official"]
            print(f"  {ours} weights:", x.get("max_abs_diff", x.get("error")), flush=True)
    # DINOv2: our wrapper (timm, img_size=224) vs the official hub model on the same images
    if images is not None and len(images):
        for ours, hub in (("dinov2_b", "dinov2_vitb14"), ("dinov2_l", "dinov2_vitl14")):
            try:
                enc = E.build(ours, CFG.VISION_ENCODERS[ours], device)
                x = torch.from_numpy(np.ascontiguousarray(images))
                f_ours = enc(x)
                off = _dinov2_official(hub).eval().to(device)
                mean = torch.tensor(enc.mean, device=device).view(1, 3, 1, 1)
                std = torch.tensor(enc.std, device=device).view(1, 3, 1, 1)
                with torch.no_grad():
                    xi = (x.to(device).permute(0, 3, 1, 2).float() / 255.0 - mean) / std
                    f_off = off(xi).float().cpu().numpy()            # class token, pos-embed interpolated
                out[f"{ours}_vs_official"] = {"n_images": int(len(images)),
                                              "cka": float(linear_cka(f_ours.astype(np.float64), f_off.astype(np.float64))),
                                              "mean_cosine": float(np.mean(np.sum(f_ours * f_off, 1) /
                                                     (np.linalg.norm(f_ours, axis=1) * np.linalg.norm(f_off, axis=1)))),
                                              "info": {k: enc.info.get(k) for k in ("resized_from", "dynamic_from")}}
                del enc, off
            except Exception as e:
                out[f"{ours}_vs_official"] = {"error": f"{type(e).__name__}: {e}"}
            if verbose: print(f"  {ours}:", out[f"{ours}_vs_official"].get("cka", out[f"{ours}_vs_official"].get("error")), flush=True)
    return out


# ---------------------------------------------------------------- driver
def run_all(rv_json_path=None, results_dir=None, nested=True, c_sens=True, validate=True, verbose=True):
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    rd = results_dir or CFG.RESULTS_DIR
    recs = RV.load(rd)
    rvj = json.load(open(rv_json_path or os.path.join(rd, "reviewer", "reviewer_analyses.json")))
    RV.check_reconstruction(recs, rd)
    out = {"status": EXPLORATORY, "generated": time.strftime("%Y-%m-%d %H:%M:%S")}
    out["S1_pilot_overlap_exclusion"] = pilot_overlap_exclusion(recs)
    out["S2_gated_rule"] = gated_rule(recs, rvj, rd)
    out["S5_association_consistency"] = association_consistency(rvj)
    out["S6_multiplicity"] = multiplicity(rvj)
    out["S7_primary_pairs"] = primary_pairs(recs)
    if verbose:
        print("S1, S2, S5, S6, S7: done", flush=True)
    if nested:
        out["S3_nested_regret"] = nested_regret(recs, rd, verbose=verbose)
    if c_sens:
        out["S4_c_sensitivity"] = c_sensitivity(recs, results_dir=rd, verbose=verbose)
    if validate:
        imgs = None
        try:
            import imgcache as IC
            arr, _ = IC.ensure_images("dtd")
            imgs = np.asarray(arr[:256])
        except Exception as e:
            print("  (DINOv2 comparison skipped: no DTD image cache:", e, ")")
        out["S8_encoder_validation"] = encoder_validation(imgs, verbose=verbose)
    return out

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/reviewer2_report.py
"""Markdown report for reviewer2.run_all() output. Everything is EXPLORATORY."""

from __future__ import annotations

from reviewer_report import ci, f, p, table

SUBS = ["all", "pathology", "vision", "text", "image"]


def build_md(o):
    L = ["# Second-round reviewer analyses (EXPLORATORY)", "",
         "Post-registration. The code first re-asserts that the saved logits reproduce every registered utility.", ""]
    s1 = o["S1_pilot_overlap_exclusion"]
    L += ["## S1. Without the six pilot datasets (Q1)", "", f"Excluded: {', '.join(s1['excluded'])}.", ""]
    rows = []
    for s in SUBS:
        v = s1.get(s)
        if not v or not v.get("pools"):
            continue
        pr, a, t = v["primary"] or {}, v["all_pairs"] or {}, v["tost_all_pairs"] or {}
        rg = v["regret"]
        rows.append([s, v["pools"], f(pr.get("estimate")) + " " + ci(pr), a.get("n", "—"), f(a.get("estimate")) + " " + ci(a),
                     t.get("tightest_margin_cleared") or "none", f(rg.get("lowest_cka"), sign=False),
                     f(rg.get("competence"), sign=False), f(rg.get("unique"), sign=False)])
    L.append(table(["subset", "pools", "D primary [95% CI]", "pairs", "D all pairs [95% CI]", "TOST",
                    "regret lowest CKA", "regret competence", "regret unique"], rows))

    s2 = o["S2_gated_rule"]
    L += ["## S2. Hybrid rules: all rules tried, gate width, paired tests (concern 3, Q2)", "",
          "Rules tried in the first exploratory round (complete list, fixed in code before it ran): "
          + ", ".join(s2["rules_tried_round1"]) + ".", "",
          "Cross-fitted regret of the gated rule by gate width (points). Per-fold selection can beat the "
          "mean-based oracle on a fold, so regret can be slightly negative.", ""]
    rows = [[s] + [f(v[g], sign=False) for g in ("0.5", "1.0", "1.5", "3.0")]
            for s, v in s2["gate_width_crossfit"].items()]
    L.append(table(["subset", "gate 0.5", "gate 1.0", "gate 1.5", "gate 3.0"], rows))
    L.append("Paired, cross-fitted: regret of competence minus regret of the rule (positive = rule better):\n")
    rows = []
    for s, d in s2["paired_vs_competence_crossfit"].items():
        for rule, x in d.items():
            rows.append([s, rule, f(x["mean"]), ci(x), p(x.get("p_t")), p(x.get("p_wilcoxon")),
                         x["better_in"], x["worse_in"], x["tie"]])
    L.append(table(["subset", "rule", "mean", "95% CI", "p (t)", "p (Wilcoxon)", "better", "worse", "tie"], rows))

    if "S3_nested_regret" in o:
        s3 = o["S3_nested_regret"]
        L += ["## S3. Fully nested selection (concern 4, Q2)", "",
              f"Selection statistics from probes refitted by inner {s3['inner_k']}-fold CV on each outer training split; "
              f"no label of the evaluation fold is used in selection. Pools with n ≤ {s3['n_max']}: "
              + ", ".join(x["pool"] for x in s3["per_pool"]) + ".", ""]
        rows = []
        for s, v in s3["summary"].items():
            g, c = v["gated_vs_competence"], v["competence_vs_lowest_cka"]
            rows.append([s, v["pools"], f(v["lowest_cka"], sign=False), f(v["competence"], sign=False),
                         f(v["unique"], sign=False), f(v["disagreement"], sign=False), f(v["gated_1.5"], sign=False),
                         f(v["rank_sum"], sign=False), f(g["mean"]) + " " + ci(g) + f" (p {p(g.get('p_t'))})",
                         f(c["mean"]) + " " + ci(c) + f" (p {p(c.get('p_t'))})"])
        L.append(table(["subset", "pools", "lowest CKA", "competence", "unique", "disagreement", "gated 1.5", "rank sum",
                        "gated vs competence", "competence vs lowest CKA"], rows))

    if "S4_c_sensitivity" in o:
        L += ["## S4. Probe regularisation C (concern 9)", "", "Same folds and matched pairs; C = 1 is registered "
              "(asserted to reproduce the registered utilities).", ""]
        rows = []
        for pool, d in o["S4_c_sensitivity"]["pools"].items():
            for C, v in d.items():
                rows.append([pool, C, f(v["mean_indiv_acc"], sign=False), v["matched_pairs"], f(v["matched_D_mean"]),
                             f(v["regret_lowest_cka"], sign=False), f(v["regret_competence"], sign=False)])
        L.append(table(["pool", "C", "mean candidate accuracy", "matched pairs", "mean D",
                        "regret lowest CKA", "regret competence"], rows))

    s5 = o["S5_association_consistency"]
    L += ["## S5. Is the weak association consistent across pools? (Q4)", ""]
    rows = []
    for m, v in s5["per_pool_sign"].items():
        lo = s5["leave_one_pool_out"][m]
        rows.append([m, f"{v['negative_in']} of {v['pools']}", f"[{f(lo['coef_range'][0])}, {f(lo['coef_range'][1])}]",
                     f"[{p(lo['p_range'][0])}, {p(lo['p_range'][1])}]", lo["most_influential"]])
    L.append(table(["measure", "pools with negative slope", "LOO coefficient range", "LOO p range",
                    "pool whose removal weakens it most"], rows))

    s6 = o["S6_multiplicity"]
    L += ["## S6. Multiplicity (concern 2)", "", s6["note"] + ".", ""]
    for fam in ("table4_all_pools", "R7_all_subsets", "D_robustness"):
        rows = [[x["test"], p(x["p"]), p(x["q_bh"]), p(x["p_holm"])] for x in s6[fam]]
        L.append(f"**{fam}** (smallest BH q = {p(s6[fam + '_min_q'])})\n")
        L.append(table(["test", "p", "q (BH)", "p (Holm)"], rows))

    L += ["## S7. Primary pairs: signed competence gap and fold interval (minor point)", ""]
    rows = [[x["pool"], f"{x['high']} vs {x['low']}", f(x["cka_gap"], sign=False), f(x["signed_gap"]), f(x["D"]),
             f"[{f(x['fold_ci_low'])}, {f(x['fold_ci_high'])}]"] for x in o["S7_primary_pairs"]]
    L.append(table(["pool", "higher vs lower CKA", "CKA gap", "signed gap (pts)", "D", "fold 95% CI"], rows))

    if "S8_encoder_validation" in o:
        v = o["S8_encoder_validation"]
        L += ["## S8. Encoder validation (concern 9)", ""]
        rows = []
        for k in ("lunit_vits16", "kaiko_vitb16"):
            x = v.get(k, {})
            rows.append([k, x.get("source", "—"), f"{x.get('same_shape', '—')} / {x.get('tensors_a', '—')}",
                         x.get("max_abs_diff", x.get("error", "—"))])
        L.append(table(["encoder", "official source", "tensors compared", "max abs difference (port vs official)"], rows))
        rows = []
        for k in ("dinov2_b_vs_official", "dinov2_l_vs_official"):
            x = v.get(k)
            if x:
                rows.append([k, x.get("n_images", "—"), f(x.get("cka"), 5, False), f(x.get("mean_cosine"), 5, False),
                             x.get("error", "")])
        if rows:
            L.append(table(["comparison", "images", "CKA (ours vs official)", "mean cosine", "error"], rows))
    return "\n".join(L)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/reviewer.py
"""Exploratory analyses requested in review (all POST-REGISTRATION).

Everything here reads saved artefacts only: the per-pool records and held-out
logits written by notebook 04 and the frozen features written by notebooks 01-03.
No probe is refitted, so no GPU is needed. Wherever a registered quantity is
re-derived (per-fold utilities, D of every pair) it is asserted equal to the
registered value first, so a reader can trust that the new analyses run on the
same numbers as the paper.

Sections (keys of the returned dict) and the review point each answers:
  R1_competence_gap   point 2 / Q1  signed competence gaps; gap-adjusted D
  R2_caliper          point 2       calipers 0.5 / 1.0 / 1.5 / 3.0 points
  R3_strong           point 3 / Q2  pairs whose candidates are within 3 pts of the anchor
  R4_dependence       point 4       two-way clustering (pool x encoder pair), leave-one-encoder-out
  R5_regret_crossfit  point 1       cross-fitted selection; hybrid rules; random baseline
  R6_algorithm1       point 1 / Q3  Algorithm 1 subsets vs size-matched competence subsets
  R7_similarity       points 5, 6 / Q4  CKA vs base ensemble, debiased and RBF CKA,
                                    mutual-kNN, logit-level CKA, on the real pools
  R8_metrics          point 8       D under macro-F1 and NLL; per-pool fold-level CIs
  R9_unmatched        point 9       matched vs unmatched comparison across all 20 pools
"""

from __future__ import annotations

import glob
import json
import os
import time

import warnings

import numpy as np
from scipy import stats

import config as CFG
import fusion
from inference import cluster_robust_ols, tost_equivalence

SUBSETS = {"all": ("pathology", "vision", "text"), "pathology": ("pathology",),
           "vision": ("vision",), "text": ("text",), "image": ("pathology", "vision")}
ORDER = [ds for dom in ("pathology", "vision", "text") for ds in CFG.DOMAINS[dom][0]]
EXPLORATORY = "EXPLORATORY (post-registration; not part of the registered analysis)"


# ---------------------------------------------------------------- loading
def load(results_dir=None):
    rd = results_dir or CFG.RESULTS_DIR
    recs = {}
    for f in glob.glob(os.path.join(rd, "pools", "*.json")):
        with open(f) as fh:
            r = json.load(fh)
        recs[r["pool"]] = r
    order = [p for p in ORDER if p in recs] + sorted(p for p in recs if p not in ORDER)
    return [recs[p] for p in order]


def load_logits(rec, results_dir=None):
    p = os.path.join(results_dir or CFG.RESULTS_DIR, "pools", f"{rec['pool']}_logits.npz")
    z = np.load(p)
    Z = {k[2:]: z[k] for k in z.files if k.startswith("Z_")}
    return Z, z["y"], z["masks"].astype(bool)


def _subset(recs, name):
    return [r for r in recs if r["domain"] in SUBSETS[name]]


def _cr(y, cl, x=None):
    """Cluster-robust mean (or regression) as a flat dict; None if not estimable."""
    y, cl = np.asarray(y, float), np.asarray(cl)
    if len(y) < 2 or len(np.unique(cl)) < 2:
        return None
    r = cluster_robust_ols(y, cl, x=x)
    i = 0 if x is None else slice(None)
    out = {"n": int(len(y)), "G": int(len(np.unique(cl))), "df": int(r["df"])}
    for k in ("estimate", "se", "p", "ci_low", "ci_high"):
        v = np.atleast_1d(r[k])
        out[k] = [float(t) for t in v] if x is not None else float(v[0])
    return out


def _tost(y, cl, margins=(1.0, 0.5, 0.25)):
    if len(y) < 2 or len(np.unique(cl)) < 2:
        return None
    res = {m: tost_equivalence(np.asarray(y, float), np.asarray(cl), margin=m) for m in margins}
    cleared = [m for m in margins if res[m]["equivalent"]]
    return {"tightest_margin_cleared": min(cleared) if cleared else None,
            **{f"p_tost_{m}": float(res[m]["p_tost"]) for m in margins}}


# ---------------------------------------------------------------- consistency
def check_reconstruction(recs, results_dir=None):
    """Recompute every candidate's per-fold primary-rule utility from the saved
    logits and assert it equals the registered record."""
    worst = 0.0
    for r in recs:
        Z, y, masks = load_logits(r, results_dir)
        base = list(r["base"])
        reg = r["utility_per_fold"]["mean_logits"]
        for c, ufold in reg.items():
            for f, m in enumerate(masks):
                acc_b = (fusion.fuse([Z[b][m] for b in base], "mean_logits").argmax(1) == y[m]).mean() * 100
                acc_c = (fusion.fuse([Z[b][m] for b in base + [c]], "mean_logits").argmax(1) == y[m]).mean() * 100
                worst = max(worst, abs((acc_c - acc_b) - ufold[f]))
    assert worst < 1e-9, f"saved logits do not reproduce the registered utilities (max diff {worst})"
    return worst


# ---------------------------------------------------------------- R1, R2, R3, R9
def _pairs(recs, cats=("matched",), caliper=None):
    out = []
    for r in recs:
        for p in r["pairs"]:
            if p["category"] not in cats or not np.isfinite(p["diff"]):
                continue
            if caliper is not None and p["comp_gap"] > caliper + 1e-12:
                continue
            q = dict(p)
            q["signed_gap"] = p["acc_high"] - p["acc_low"]          # >0: higher-CKA is more accurate
            q["anchor_acc"] = r["indiv_acc"][r["anchor"]]
            q["domain"] = r["domain"]
            out.append(q)
    return out


def _primary(pairs):
    best = {}
    for p in pairs:
        if p["pool"] not in best or p["cka_gap"] > best[p["pool"]]["cka_gap"]:
            best[p["pool"]] = p
    return list(best.values())


def r1_competence_gap(recs):
    out = {"note": EXPLORATORY}
    for s in SUBSETS:
        P = _pairs(_subset(recs, s))
        if not P:
            continue
        g = np.array([p["signed_gap"] for p in P]); d = np.array([p["diff"] for p in P])
        cl = np.array([p["pool"] for p in P])
        prim = _primary(P)
        gp = np.array([p["signed_gap"] for p in prim]); dp = np.array([p["diff"] for p in prim])
        clp = np.array([p["pool"] for p in prim])
        adj = _cr(d, cl, x=g)
        adjp = _cr(dp, clp, x=gp) if len(prim) > 2 else None
        out[s] = {
            "pairs": len(P),
            "signed_gap_mean": float(g.mean()), "signed_gap_sd": float(g.std(ddof=1)) if len(g) > 1 else None,
            "frac_higher_cka_more_accurate": float((g > 0).mean()),
            "signed_gap_cluster_robust": _cr(g, cl),
            "D_adjusted_all_pairs": None if adj is None else
                {"intercept_D_at_zero_gap": adj["estimate"][0], "ci": [adj["ci_low"][0], adj["ci_high"][0]],
                 "p": adj["p"][0], "slope_pp_per_gap_point": adj["estimate"][1], "slope_p": adj["p"][1],
                 "G": adj["G"], "n": adj["n"]},
            "D_adjusted_primary": None if adjp is None else
                {"intercept_D_at_zero_gap": adjp["estimate"][0], "ci": [adjp["ci_low"][0], adjp["ci_high"][0]],
                 "p": adjp["p"][0], "G": adjp["G"]},
        }
    return out


def r2_caliper(recs, calipers=(0.5, 1.0, 1.5, 3.0)):
    out = {"note": EXPLORATORY + "; 1.5 is the registered caliper",
           "eligible": "pairs with CKA gap >= 0.15 and separated by > 3 median subsampling SDs"}
    for s in SUBSETS:
        out[s] = {}
        for c in calipers:
            P = _pairs(_subset(recs, s), cats=("matched", "unmatched"), caliper=c)
            if not P:
                out[s][str(c)] = None
                continue
            d = [p["diff"] for p in P]; cl = [p["pool"] for p in P]
            prim = _primary(P)
            out[s][str(c)] = {"all_pairs": _cr(d, cl), "primary": _cr([p["diff"] for p in prim],
                                                                     [p["pool"] for p in prim]),
                              "tost_all_pairs": _tost(d, cl)}
    return out


def r3_strong(recs, within=3.0):
    out = {"note": EXPLORATORY, "rule": f"both candidates within {within} points of the anchor's accuracy"}
    for s in SUBSETS:
        P = [p for p in _pairs(_subset(recs, s))
             if p["anchor_acc"] - min(p["acc_high"], p["acc_low"]) <= within]
        if not P:
            out[s] = None
            continue
        d = [p["diff"] for p in P]; cl = [p["pool"] for p in P]
        out[s] = {"all_pairs": _cr(d, cl), "tost_all_pairs": _tost(d, cl),
                  "pairs": len(P), "pools": len(set(cl)),
                  "mean_utility_of_members": float(np.mean(
                      [u for r in _subset(recs, s) for c in r["candidates"]
                       for u in [c["utility"]] if any(c["name"] in (p["high"], p["low"]) and p["pool"] == r["pool"]
                                                      for p in P)]))}
    # how many matched pairs have both candidates weaker than the base fusion
    allp = _pairs(recs)
    weak = 0
    for p in allp:
        r = next(r for r in recs if r["pool"] == p["pool"])
        base_acc = float(np.mean(r["base_fold_acc"]["mean_logits"]))
        weak += (p["acc_high"] < base_acc) and (p["acc_low"] < base_acc)
    out["matched_pairs_both_weaker_than_base_fusion"] = {"count": int(weak), "of": len(allp)}
    return out


def r9_unmatched(recs):
    out = {"note": EXPLORATORY}
    for s in SUBSETS:
        U = _pairs(_subset(recs, s), cats=("unmatched",))
        M = _pairs(_subset(recs, s))
        if not U:
            out[s] = None
            continue
        d = np.array([p["diff"] for p in U]); g = np.array([p["signed_gap"] for p in U])
        cl = np.array([p["pool"] for p in U])
        reg = _cr(d, cl, x=g)
        out[s] = {"matched_all_pairs": _cr([p["diff"] for p in M], [p["pool"] for p in M]) if M else None,
                  "unmatched_all_pairs": _cr(d, cl),
                  "unmatched_frac_higher_cka_more_accurate": float((g > 0).mean()),
                  "unmatched_D_on_signed_gap": None if reg is None else
                      {"slope_pp_per_point": reg["estimate"][1], "p": reg["p"][1],
                       "intercept": reg["estimate"][0], "intercept_p": reg["p"][0]},
                  "unmatched_pairs": int(len(U))}
    return out


# ---------------------------------------------------------------- R4
def _two_way(y, c1, c2):
    """Intercept-only mean with Cameron-Gelbach-Miller two-way clustered SE."""
    y = np.asarray(y, float)
    e = y - y.mean()
    n = len(y)

    def V(cl):
        cl = np.asarray(cl)
        u = np.unique(cl)
        G = len(u)
        s = sum(e[cl == g].sum() ** 2 for g in u)
        return G / max(G - 1, 1) * s / n ** 2, G

    v1, g1 = V(c1)
    v2, g2 = V(c2)
    v12, _ = V([f"{a}||{b}" for a, b in zip(c1, c2)])
    var = max(v1 + v2 - v12, v1, v2)          # guard against a negative CGM variance
    se = np.sqrt(var)
    df = min(g1, g2) - 1
    tc = stats.t.ppf(0.975, df)
    m = float(y.mean())
    return {"estimate": m, "se": float(se), "ci_low": m - tc * se, "ci_high": m + tc * se,
            "p": float(2 * stats.t.sf(abs(m / se), df)) if se > 0 else None,
            "G_pool": g1, "G_encoder_pair": g2, "df": df}


def r4_dependence(recs):
    out = {"note": EXPLORATORY}
    for s in SUBSETS:
        P = _pairs(_subset(recs, s))
        if len(P) < 3:
            out[s] = None
            continue
        y = [p["diff"] for p in P]
        pool = [p["pool"] for p in P]
        pair = ["|".join(sorted((p["high"], p["low"]))) for p in P]
        enc = sorted({e for p in P for e in (p["high"], p["low"])})
        loo = []
        for e in enc:
            Q = [p for p in P if e not in (p["high"], p["low"])]
            r = _cr([p["diff"] for p in Q], [p["pool"] for p in Q])
            if r:
                loo.append({"dropped_encoder": e, **{k: r[k] for k in ("estimate", "ci_low", "ci_high", "p", "n", "G")}})
        prim = _primary(P)
        out[s] = {"pool_clustered": _cr(y, pool),
                  "two_way_pool_x_encoder_pair": _two_way(y, pool, pair),
                  "encoder_pair_clustered": _cr(y, pair),
                  "primary_two_way": _two_way([p["diff"] for p in prim], [p["pool"] for p in prim],
                                              ["|".join(sorted((p["high"], p["low"]))) for p in prim])
                  if len(prim) > 2 else None,
                  "leave_one_encoder_out": loo,
                  "loo_range": [min(l["estimate"] for l in loo), max(l["estimate"] for l in loo)] if loo else None,
                  "loo_all_ci_contain_zero": all(l["ci_low"] <= 0 <= l["ci_high"] for l in loo) if loo else None}
    return out


# ---------------------------------------------------------------- R5
RULE_DESCR = {
    "lowest_cka": "lowest CKA with the anchor (label-free; the registered heuristic)",
    "competence": "highest accuracy",
    "unique": "largest unique contribution over the base",
    "disagreement": "highest disagreement with the base",
    "gated_diversity": "lowest CKA among candidates within 1.5 points of the most accurate",
    "rank_sum": "best sum of accuracy rank and (inverse) CKA rank; ties to accuracy",
    "random": "uniform random candidate (expected value)",
}


def _select(rule, names, acc, cka, uniq, dis):
    # Registered rules break ties exactly as the registered analysis does: the
    # first candidate in record order wins (Python's min/max on the value alone).
    if rule == "lowest_cka":
        return min(names, key=lambda c: cka[c])
    if rule == "competence":
        return max(names, key=lambda c: acc[c])
    if rule == "unique":
        return max(names, key=lambda c: uniq[c])
    if rule == "disagreement":
        return max(names, key=lambda c: dis[c])
    if rule == "gated_diversity":
        top = max(acc.values())
        ok = [c for c in names if acc[c] >= top - 1.5]
        return min(ok, key=lambda c: (cka[c], -acc[c]))
    if rule == "rank_sum":
        ra = {c: i for i, c in enumerate(sorted(names, key=lambda c: -acc[c]))}
        rc = {c: i for i, c in enumerate(sorted(names, key=lambda c: cka[c]))}
        return min(names, key=lambda c: (ra[c] + rc[c], ra[c]))
    raise KeyError(rule)


def r5_regret_crossfit(recs, results_dir=None):
    """For each pool and evaluation fold f, every selection statistic that uses
    labels (accuracy, unique contribution, disagreement) is computed on the
    held-out rows of the OTHER folds only; the chosen candidate is scored by its
    utility on fold f. CKA uses no labels and is used as registered.

    Caveat stated in the output: each fold's held-out logits come from a probe
    trained on the other folds, so selection statistics on folds != f still depend
    on fold-f labels through those probes. Removing that requires refitting probes
    in a nested design (GPU); this analysis removes the direct reuse."""
    rules = [r for r in RULE_DESCR if r != "random"]
    per_pool = []
    for r in recs:
        Z, y, masks = load_logits(r, results_dir)
        base = list(r["base"])
        cands = [c["name"] for c in r["candidates"]]
        U = {c: np.array(r["utility_per_fold"]["mean_logits"][c]) for c in cands}
        oracle = max(U[c].mean() for c in cands)
        cka = {c: r["cka_with_anchor"][c] for c in cands}
        row = {"pool": r["pool"], "domain": r["domain"], "oracle_mean_utility": float(oracle),
               "random_regret": float(oracle - np.mean([U[c].mean() for c in cands]))}
        for rule in rules:
            got_cf, got_in = [], []
            for f, m in enumerate(masks):
                tr = ~m
                pred = {c: Z[c].argmax(1) for c in cands + base}
                basep = fusion.fuse([Z[b] for b in base], "mean_logits").argmax(1)
                base_any = np.zeros(len(y), bool)
                for b in base:
                    base_any |= pred[b] == y
                def stats_on(rows):
                    acc = {c: float((pred[c][rows] == y[rows]).mean() * 100) for c in cands}
                    uniq = {c: float(((pred[c][rows] == y[rows]) & ~base_any[rows]).mean() * 100) for c in cands}
                    dis = {c: float((pred[c][rows] != basep[rows]).mean()) for c in cands}
                    return acc, uniq, dis
                a_cf, u_cf, d_cf = stats_on(tr)
                got_cf.append(U[_select(rule, cands, a_cf, cka, u_cf, d_cf)][f])
            # in-sample version, as registered (statistics on all rows)
            a_in = {c: r["indiv_acc"][c] for c in cands}
            u_in = {c: next(x["headroom_contrib"] for x in r["candidates"] if x["name"] == c) for c in cands}
            d_in = {c: next(x["disagreement"] for x in r["candidates"] if x["name"] == c) for c in cands}
            pick = _select(rule, cands, a_in, cka, u_in, d_in)
            row[f"{rule}_regret_crossfit"] = float(oracle - np.mean(got_cf))
            row[f"{rule}_regret_insample"] = float(oracle - U[pick].mean())
        per_pool.append(row)
    bp = os.path.join(results_dir or CFG.RESULTS_DIR, "battery.json")
    if os.path.exists(bp):
        bat = json.load(open(bp))
        regname = {"lowest_cka": "cka_diversity", "competence": "competence",
                   "unique": "oracle_headroom", "disagreement": "disagreement_with_base"}
        for s in SUBSETS:
            reg = (bat.get(s, {}).get("D_selection_regret") or {}).get("per_rule", {})
            rows = [p for p in per_pool if p["domain"] in SUBSETS[s]]
            for mine, theirs in regname.items():
                if theirs in reg and rows:
                    v = float(np.mean([p[f"{mine}_regret_insample"] for p in rows]))
                    assert abs(v - reg[theirs]["mean_regret_pp"]) < 1e-9, \
                        f"in-sample {mine} regret {v} != registered {reg[theirs]['mean_regret_pp']} ({s})"
    out = {"note": EXPLORATORY,
           "caveat": " ".join(r5_regret_crossfit.__doc__.split("Caveat stated in the output: ")[1].split()),
           "registered_rules_reproduced_in_sample": os.path.exists(bp),
           "rules": RULE_DESCR, "per_pool": per_pool}
    for s in SUBSETS:
        rows = [p for p in per_pool if p["domain"] in SUBSETS[s]]
        if not rows:
            continue
        summ = {"pools": len(rows), "random_regret": float(np.mean([p["random_regret"] for p in rows]))}
        for rule in rules:
            for kind in ("crossfit", "insample"):
                v = np.array([p[f"{rule}_regret_{kind}"] for p in rows])
                summ[f"{rule}_{kind}"] = float(v.mean())
            dcf = np.array([p[f"lowest_cka_regret_crossfit"] - p[f"{rule}_regret_crossfit"] for p in rows])
            if rule != "lowest_cka" and len(rows) > 1:
                t = stats.ttest_1samp(dcf, 0.0)
                summ[f"{rule}_vs_lowest_cka_crossfit"] = {
                    "mean_regret_saved_pp": float(dcf.mean()), "p_t": float(t.pvalue),
                    "rule_better_in": int((dcf > 1e-9).sum()), "worse_in": int((dcf < -1e-9).sum())}
        out[s] = summ
    return out


# ---------------------------------------------------------------- R6 (Algorithm 1)
def _alg1(names, C, u, th_cka, th_u):
    keep = set(names)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            if a in keep and b in keep and C[(a, b)] > th_cka:
                weak = a if u[a] < u[b] else b
                if u[weak] < th_u:
                    keep.discard(weak)
    return sorted(keep)


def r6_algorithm1(recs, sim, results_dir=None, th_ckas=(0.6, 0.7, 0.8, 0.9), th_us=(0.5, 1.0, 2.0)):
    """Algorithm 1 (paper appendix) applied to every pool's full member set,
    cross-fitted: unique contributions and accuracies for fold f are computed on
    the other folds' held-out rows; CKA (label-free) from the feature matrices.
    Compared against the size-matched subset of the most accurate members and
    against fusing all members. Fusion: the registered primary rule."""
    rows = []
    for r in recs:
        if r["pool"] not in sim:
            continue
        Z, y, masks = load_logits(r, results_dir)
        names = sorted(Z)
        C = sim[r["pool"]]["pairwise_linear_cka"]
        for thc in th_ckas:
            for thu in th_us:
                acc_alg, acc_top, acc_all, sizes = [], [], [], []
                for f, m in enumerate(masks):
                    tr = ~m
                    pred = {c: Z[c].argmax(1) for c in names}
                    corr = {c: pred[c][tr] == y[tr] for c in names}
                    others_any = {c: np.zeros(tr.sum(), bool) for c in names}
                    for c in names:
                        for o in names:
                            if o != c:
                                others_any[c] |= corr[o]
                    u = {c: float((corr[c] & ~others_any[c]).mean() * 100) for c in names}
                    acc = {c: float(corr[c].mean() * 100) for c in names}
                    S = _alg1(names, C, u, thc, thu)
                    k = len(S)
                    T = sorted(names, key=lambda c: -acc[c])[:k]
                    score = lambda M: float((fusion.fuse([Z[c][m] for c in M], "mean_logits").argmax(1)
                                             == y[m]).mean() * 100)
                    acc_alg.append(score(S)); acc_top.append(score(T)); acc_all.append(score(names))
                    sizes.append(k)
                rows.append({"pool": r["pool"], "domain": r["domain"], "theta_cka": thc, "theta_unique": thu,
                             "mean_size": float(np.mean(sizes)), "n_members": len(names),
                             "acc_alg1": float(np.mean(acc_alg)), "acc_topk_competence": float(np.mean(acc_top)),
                             "acc_all_members": float(np.mean(acc_all))})
    out = {"note": EXPLORATORY, "per_pool": rows, "summary": []}
    for thc in th_ckas:
        for thu in th_us:
            R = [x for x in rows if x["theta_cka"] == thc and x["theta_unique"] == thu]
            if len(R) < 2:
                continue
            d = np.array([x["acc_alg1"] - x["acc_topk_competence"] for x in R])
            t = stats.ttest_1samp(d, 0.0) if np.any(d != 0) else None
            out["summary"].append({"theta_cka": thc, "theta_unique": thu, "pools": len(R),
                                   "mean_size": float(np.mean([x["mean_size"] for x in R])),
                                   "alg1_minus_topk_pp": float(d.mean()),
                                   "p_t": float(t.pvalue) if t is not None else None,
                                   "alg1_worse_in": int((d < -1e-9).sum()), "alg1_better_in": int((d > 1e-9).sum()),
                                   "alg1_minus_all_pp": float(np.mean([x["acc_alg1"] - x["acc_all_members"] for x in R]))})
    return out


# ---------------------------------------------------------------- R7 (similarity measures)
def _center(X):
    return X - X.mean(0, keepdims=True)


def _lin_cka(X, Y):
    X, Y = _center(X), _center(Y)
    num = np.linalg.norm(Y.T @ X) ** 2
    return float(num / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)))


def _unbiased_hsic(K, L):
    n = K.shape[0]
    K = K.copy(); L = L.copy()
    np.fill_diagonal(K, 0); np.fill_diagonal(L, 0)
    t1 = np.sum(K * L)
    t2 = K.sum() * L.sum() / ((n - 1) * (n - 2))
    t3 = 2 * (K.sum(0) @ L.sum(0)) / (n - 2)
    return (t1 + t2 - t3) / (n * (n - 3))


def _debiased_cka(X, Y):
    X, Y = _center(X), _center(Y)
    K, L = X @ X.T, Y @ Y.T
    kl = _unbiased_hsic(K, L)
    return float(kl / np.sqrt(max(_unbiased_hsic(K, K), 1e-300) * max(_unbiased_hsic(L, L), 1e-300)))


def _rbf_cka(X, Y):
    def gram(A):
        sq = np.sum(A * A, 1)
        D = np.maximum(sq[:, None] + sq[None, :] - 2 * A @ A.T, 0)
        sig2 = np.median(D[np.triu_indices_from(D, 1)])
        return np.exp(-D / max(sig2, 1e-12))
    K, L = gram(X), gram(Y)
    n = K.shape[0]
    H = np.eye(n) - 1.0 / n
    Kc, Lc = H @ K @ H, H @ L @ H
    return float(np.sum(Kc * Lc) / (np.linalg.norm(Kc) * np.linalg.norm(Lc)))


def _mutual_knn(X, Y, k=10):
    from sklearn.neighbors import NearestNeighbors
    def nn(A):
        A = A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-12)
        idx = NearestNeighbors(n_neighbors=k + 1).fit(A).kneighbors(A, return_distance=False)
        return idx[:, 1:]
    a, b = nn(X), nn(Y)
    return float(np.mean([len(set(a[i]) & set(b[i])) / k for i in range(len(a))]))


def _unit(X):
    X = _center(np.asarray(X, np.float64))
    return X / max(np.linalg.norm(X), 1e-12)


def similarity_measures(recs, n_sub=3000, n_sub_rbf=2000, results_dir=None, verbose=True):
    """Per pool and candidate: several similarity measures to the anchor and to the
    base ensemble, computed on one fixed subsample of rows (seed 0)."""
    import extract as X_
    out = {}
    for r in recs:
        ds, t0 = r["pool"], time.time()
        man = json.load(open(os.path.join(CFG.MANIFEST_DIR, f"{ds}.json")))
        names = sorted(r["indiv_acc"])
        n = man["n"]
        idx = np.sort(np.random.default_rng(0).choice(n, min(n_sub, n), replace=False))
        idx_r = idx[np.sort(np.random.default_rng(1).choice(len(idx), min(n_sub_rbf, len(idx)), replace=False))]
        F = {}
        for e in names:
            p = X_._paths(ds, e)[1]
            if not os.path.exists(p):
                break
            F[e] = np.asarray(np.load(p)[idx], np.float64)    # one sequential read; scattered reads over Drive are slow
        if len(F) != len(names):
            if verbose:
                print(f"  [{ds}] features missing; skipped")
            continue
        Z, y, masks = load_logits(r, results_dir)
        anchor, base = r["anchor"], list(r["base"])
        Fbase = np.concatenate([_unit(F[b]) for b in base], 1)
        Zbase = np.concatenate([_unit(Z[b]) for b in base], 1)
        pos = {e: i for i, e in enumerate(idx)}
        row_r = np.array([pos[i] for i in idx_r])
        C = {}
        for i, a in enumerate(names):
            for b in names[i + 1:]:
                C[(a, b)] = C[(b, a)] = _lin_cka(F[a], F[b])
        cand = {}
        for c in [x["name"] for x in r["candidates"]]:
            cand[c] = {
                "linear_cka_anchor_subsample": C[(c, anchor)],
                "linear_cka_base": _lin_cka(F[c], Fbase),
                "debiased_cka_anchor": _debiased_cka(F[c], F[anchor]),
                "rbf_cka_anchor": _rbf_cka(F[c][row_r], F[anchor][row_r]),
                "mutual_knn_anchor": _mutual_knn(F[c], F[anchor]),
                "logit_cka_anchor": _lin_cka(Z[c], Z[anchor]),
                "logit_cka_base": _lin_cka(Z[c], Zbase),
            }
        out[ds] = {"pairwise_linear_cka": C, "candidates": cand, "n_rows": int(len(idx)),
                   "n_rows_rbf": int(len(idx_r))}
        if verbose:
            print(f"  [{ds}] {len(cand)} candidates, {len(idx)} rows: {time.time() - t0:.0f}s", flush=True)
    return out


MEASURES = ["linear_cka_registered", "linear_cka_base", "debiased_cka_anchor", "rbf_cka_anchor",
            "mutual_knn_anchor", "logit_cka_anchor", "logit_cka_base"]


def r7_similarity(recs, sim):
    rows = []
    for r in recs:
        if r["pool"] not in sim:
            continue
        for c in r["candidates"]:
            m = dict(sim[r["pool"]]["candidates"][c["name"]])
            m["linear_cka_registered"] = r["cka_with_anchor"][c["name"]]
            rows.append({"pool": r["pool"], "domain": r["domain"], "name": c["name"],
                         "utility": c["utility"], "indiv_acc": c["indiv_acc"], **m})
    out = {"note": EXPLORATORY, "candidates": rows}
    for s in SUBSETS:
        R = [x for x in rows if x["domain"] in SUBSETS[s]]
        if len({x["pool"] for x in R}) < 2:
            continue
        cl = np.array([x["pool"] for x in R])
        u = np.array([x["utility"] for x in R])
        acc = np.array([x["indiv_acc"] for x in R])
        # within-pool centring of accuracy removes pool-level differences in difficulty
        accc = acc - np.array([acc[cl == p].mean() for p in cl])
        res = {}
        for meas in MEASURES:
            v = np.array([x[meas] for x in R])
            vc = v - np.array([v[cl == p].mean() for p in cl])
            reg = cluster_robust_ols(u - np.array([u[cl == p].mean() for p in cl]), cl,
                                     x=np.column_stack([vc, accc]))
            # regret of choosing the least similar candidate by this measure
            reg_regret = []
            for p in sorted(set(cl)):
                Rp = [x for x in R if x["pool"] == p]
                best = max(x["utility"] for x in Rp)
                reg_regret.append(best - min(Rp, key=lambda x: x[meas])["utility"])
            res[meas] = {"coef_pp_per_unit": float(reg["estimate"][1]), "p": float(reg["p"][1]),
                         "acc_coef": float(reg["estimate"][2]), "acc_p": float(reg["p"][2]),
                         "spearman_with_utility_within_pool": float(stats.spearmanr(vc, u - np.array(
                             [u[cl == p].mean() for p in cl])).correlation),
                         "regret_choose_least_similar_pp": float(np.mean(reg_regret)),
                         "G": int(len(set(cl))), "n": int(len(R))}
        # matched pairs re-oriented by each measure
        P = _pairs(_subset(recs, s))
        for meas in MEASURES:
            ds = []
            for p in P:
                if p["pool"] not in sim:
                    continue
                look = {x["name"]: x for x in R if x["pool"] == p["pool"]}
                if p["high"] not in look or p["low"] not in look:
                    continue
                a, b = look[p["high"]][meas], look[p["low"]][meas]
                ds.append((p["pool"], p["diff"] if a >= b else -p["diff"]))
            if len({d[0] for d in ds}) >= 2:
                res[meas]["matched_D_oriented_by_measure"] = _cr([d[1] for d in ds], [d[0] for d in ds])
        out[s] = res
    return out


# ---------------------------------------------------------------- R8
def r8_metrics(recs, results_dir=None):
    from sklearn.metrics import f1_score
    out = {"note": EXPLORATORY, "per_pool_primary": []}
    pairD = {"macro_f1": [], "nll": [], "accuracy_check": []}
    for r in recs:
        Z, y, masks = load_logits(r, results_dir)
        base = list(r["base"])
        def metr(M, m):
            zl = fusion.fuse([Z[c][m] for c in M], "mean_logits")
            pr = fusion.fuse([Z[c][m] for c in M], "mean_probs")
            f1 = f1_score(y[m], zl.argmax(1), average="macro", labels=np.unique(y)) * 100
            nll = -np.mean(np.log(np.clip(pr[np.arange(m.sum()), y[m]], 1e-12, 1)))
            acc = (zl.argmax(1) == y[m]).mean() * 100
            return f1, nll, acc
        cache = {}
        def util(c):
            if c not in cache:
                vals = []
                for f, m in enumerate(masks):
                    fb, nb, ab = metr(base, m)
                    fc, nc, ac = metr(base + [c], m)
                    vals.append((fc - fb, nb - nc, ac - ab))      # NLL utility: reduction in NLL
                cache[c] = np.array(vals)
            return cache[c]
        for p in r["pairs"]:
            if p["category"] != "matched" or not np.isfinite(p["diff"]):
                continue
            dh, dl = util(p["high"]), util(p["low"])
            dacc = (dh[:, 2] - dl[:, 2]).mean()
            assert abs(dacc - p["diff"]) < 1e-9, f"{r['pool']}: accuracy D not reproduced"
            pairD["macro_f1"].append((r["pool"], r["domain"], (dh[:, 0] - dl[:, 0]).mean(), p.get("primary")))
            pairD["nll"].append((r["pool"], r["domain"], (dh[:, 1] - dl[:, 1]).mean(), p.get("primary")))
            if p.get("primary"):
                fd = np.array(p["fold_diffs"])
                tc = stats.t.ppf(0.975, len(fd) - 1)
                se = fd.std(ddof=1) / np.sqrt(len(fd))
                out["per_pool_primary"].append({"pool": r["pool"], "domain": r["domain"], "D": float(fd.mean()),
                                                "fold_ci_low": float(fd.mean() - tc * se),
                                                "fold_ci_high": float(fd.mean() + tc * se),
                                                "high": p["high"], "low": p["low"]})
    for s in SUBSETS:
        out[s] = {}
        for k in ("macro_f1", "nll"):
            rows = [x for x in pairD[k] if x[1] in SUBSETS[s]]
            prim = [x for x in rows if x[3]]
            out[s][k] = {"all_pairs": _cr([x[2] for x in rows], [x[0] for x in rows]),
                         "primary": _cr([x[2] for x in prim], [x[0] for x in prim])}
    out["sign_note"] = ("D = metric gain of the higher-CKA minus the lower-CKA candidate; for NLL the gain is "
                        "the reduction in NLL (positive = better), so all metrics share the sign convention")
    return out


# ---------------------------------------------------------------- accuracy table
def accuracy_table(recs):
    """Held-out linear-probe accuracy of every encoder in every pool, for comparison
    with published linear-probe results (protocols differ; see the paper)."""
    rows = []
    for r in recs:
        for e, a in sorted(r["indiv_acc"].items(), key=lambda t: -t[1]):
            rows.append({"pool": r["pool"], "domain": r["domain"], "encoder": e, "accuracy": a,
                         "anchor": e == r["anchor"]})
    return rows


# ---------------------------------------------------------------- driver
def run_all(results_dir=None, with_features=True, verbose=True):
    warnings.filterwarnings("ignore", category=RuntimeWarning)   # zero-variance subsets
    rd = results_dir or CFG.RESULTS_DIR
    recs = load(rd)
    t0 = time.time()
    out = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "status": EXPLORATORY,
           "pools": [r["pool"] for r in recs]}
    out["reconstruction_max_abs_diff"] = check_reconstruction(recs, rd)
    if verbose:
        print("registered utilities reproduced from the saved logits (max diff "
              f"{out['reconstruction_max_abs_diff']:.1e})", flush=True)
    for key, fn in (("R1_competence_gap", r1_competence_gap), ("R2_caliper", r2_caliper),
                    ("R3_strong", r3_strong), ("R4_dependence", r4_dependence),
                    ("R9_unmatched", r9_unmatched)):
        out[key] = fn(recs)
        if verbose:
            print(f"{key}: done", flush=True)
    out["R5_regret_crossfit"] = r5_regret_crossfit(recs, rd)
    if verbose:
        print("R5_regret_crossfit: done", flush=True)
    out["R8_metrics"] = r8_metrics(recs, rd)
    if verbose:
        print("R8_metrics: done", flush=True)
    out["accuracy_table"] = accuracy_table(recs)
    if with_features:
        sim = similarity_measures(recs, results_dir=rd, verbose=verbose)
        out["R7_similarity"] = r7_similarity(recs, sim)
        out["R6_algorithm1"] = r6_algorithm1(recs, sim, rd)
        if verbose:
            print("R6, R7: done", flush=True)
    out["wall_minutes"] = round((time.time() - t0) / 60, 1)
    return out

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/reviewer_report.py
"""Turn reviewer.run_all() output into reviewer_analyses.md and figures.
Every table is headed EXPLORATORY: none of these analyses was registered."""

from __future__ import annotations

import os

import numpy as np

from reviewer import MEASURES, RULE_DESCR, SUBSETS

SUBS = list(SUBSETS)


def f(v, nd=2, sign=True):
    try:
        v = float(v)
    except (TypeError, ValueError):
        return "—"
    if not np.isfinite(v):
        return "—"
    return f"{v:+.{nd}f}" if sign else f"{v:.{nd}f}"


def p(v):
    try:
        v = float(v)
    except (TypeError, ValueError):
        return "—"
    return "—" if not np.isfinite(v) else (f"{v:.3f}" if v >= 0.001 else f"{v:.1e}")


def ci(d, lo="ci_low", hi="ci_high"):
    return "—" if not d or lo not in d or hi not in d else f"[{f(d[lo])}, {f(d[hi])}]"


def table(head, rows):
    L = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    L += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    return "\n".join(L) + "\n"


def build_md(out):
    L = ["# Reviewer analyses (EXPLORATORY)", "",
         "None of these analyses was part of the registration. They read only the saved per-pool records, "
         "held-out logits and frozen features; no probe was refitted. Before anything else the code asserts that "
         f"the saved logits reproduce every registered per-fold utility (max difference "
         f"{out['reconstruction_max_abs_diff']:.1e}), that every matched pair's accuracy-based D is reproduced, and "
         "that the registered selection rules give the registered regrets when applied in-sample.", ""]

    # R1
    L += ["## R1. Competence imbalance within matched pairs (review point 2, Q1)", "",
          "Signed gap = accuracy of the higher-CKA candidate minus the lower-CKA one. Gap-adjusted D is the "
          "intercept of D regressed on the signed gap (cluster-robust by pool): the difference expected at "
          "equal competence.", ""]
    rows = []
    for s in SUBS:
        r = out["R1_competence_gap"].get(s)
        if not r:
            continue
        g = r["signed_gap_cluster_robust"] or {}
        a = r["D_adjusted_all_pairs"] or {}
        rows.append([s, r["pairs"], f(r["signed_gap_mean"]), ci(g), f"{r['frac_higher_cka_more_accurate']:.0%}",
                     f(a.get("intercept_D_at_zero_gap")),
                     "—" if not a else f"[{f(a['ci'][0])}, {f(a['ci'][1])}]", p(a.get("p")),
                     f(a.get("slope_pp_per_gap_point"))])
    L.append(table(["subset", "pairs", "mean signed gap (pts)", "95% CI", "higher-CKA more accurate",
                    "gap-adjusted D", "95% CI", "p", "D per gap point"], rows))

    # R2
    L += ["## R2. Caliper sensitivity (review point 2)", "",
          "Pairs with CKA gap ≥ 0.15 (and beyond the subsampling tolerance) whose competence gap is within the "
          "caliper; 1.5 is the registered caliper.", ""]
    rows = []
    for s in SUBS:
        for c, v in (out["R2_caliper"].get(s) or {}).items():
            if not v:
                rows.append([s, c, "—", "—", "—", "—", "—", "—"])
                continue
            a, pr, t = v["all_pairs"] or {}, v["primary"] or {}, v["tost_all_pairs"] or {}
            rows.append([s, c, a.get("n", "—"), a.get("G", "—"), f(a.get("estimate")), ci(a),
                         f(pr.get("estimate")) + " " + ci(pr), t.get("tightest_margin_cleared") or "none"])
    L.append(table(["subset", "caliper (pts)", "pairs", "G", "D all pairs", "95% CI", "D primary [95% CI]",
                    "TOST (tightest margin)"], rows))

    # R3
    s3 = out["R3_strong"]
    L += ["## R3. Strong candidates only (review point 3, Q2)", "", f"Rule: {s3['rule']}.", ""]
    rows = []
    for s in SUBS:
        v = s3.get(s)
        if not v:
            rows.append([s, 0, "—", "—", "—", "—"])
            continue
        a, t = v["all_pairs"] or {}, v["tost_all_pairs"] or {}
        rows.append([s, v["pairs"], v["pools"], f(a.get("estimate")), ci(a), t.get("tightest_margin_cleared") or "none"])
    L.append(table(["subset", "pairs", "pools", "D", "95% CI", "TOST (tightest margin)"], rows))
    w = s3["matched_pairs_both_weaker_than_base_fusion"]
    L.append(f"Matched pairs in which both candidates are individually weaker than the base fusion: "
             f"{w['count']} of {w['of']}.\n")

    # R4
    L += ["## R4. Dependence between pools (review point 4)", ""]
    rows = []
    for s in SUBS:
        v = out["R4_dependence"].get(s)
        if not v:
            continue
        a, tw, ep = v["pool_clustered"] or {}, v["two_way_pool_x_encoder_pair"], v["encoder_pair_clustered"] or {}
        lr = v["loo_range"]
        rows.append([s, f(a.get("estimate")), ci(a), ci(ep), ci(tw), f"df {tw['df']}",
                     "—" if not lr else f"[{f(lr[0])}, {f(lr[1])}]", v["loo_all_ci_contain_zero"]])
    L.append(table(["subset", "D all pairs", "CI: by pool", "CI: by encoder pair", "CI: two-way",
                    "two-way df", "leave-one-encoder-out range", "every LOO CI contains 0"], rows))

    # R5
    r5 = out["R5_regret_crossfit"]
    L += ["## R5. Selection regret with cross-fitted selection and hybrid rules (review point 1)", "",
          "Label-using statistics for evaluation fold f are computed on the other folds' held-out rows only; "
          "the chosen candidate is scored on fold f. " + r5["caveat"], "",
          "Rules: " + "; ".join(f"**{k}** = {v}" for k, v in RULE_DESCR.items()) + ".", ""]
    rules = [k for k in RULE_DESCR if k != "random"]
    head = ["subset", "pools", "random"] + [f"{k} (cross-fit / in-sample)" for k in rules]
    rows = []
    for s in SUBS:
        v = r5.get(s)
        if not v:
            continue
        rows.append([s, v["pools"], f(v["random_regret"], sign=False)] +
                    [f"{f(v[k + '_crossfit'], sign=False)} / {f(v[k + '_insample'], sign=False)}" for k in rules])
    L.append(table(head, rows))
    rows = []
    for s in SUBS:
        v = r5.get(s)
        if not v:
            continue
        for k in rules:
            c = v.get(f"{k}_vs_lowest_cka_crossfit")
            if c:
                rows.append([s, k, f(c["mean_regret_saved_pp"]), p(c["p_t"]), c["rule_better_in"], c["worse_in"]])
    L.append("Regret saved relative to the lowest-CKA rule (cross-fitted; positive = the rule beats lowest CKA):\n")
    L.append(table(["subset", "rule", "regret saved (pts)", "p (paired t over pools)", "better in", "worse in"], rows))

    # R6
    if "R6_algorithm1" in out:
        L += ["## R6. Algorithm 1 on all pools, cross-fitted (review point 1, Q3)", "",
              "Accuracy of fusing the subset Algorithm 1 keeps, minus fusing the same number of members chosen by "
              "cross-fitted accuracy; and minus fusing all members. Primary fusion rule. Thresholds are a fixed grid, "
              "not tuned.", ""]
        rows = [[x["theta_cka"], x["theta_unique"], x["pools"], f(x["mean_size"], 1, False),
                 f(x["alg1_minus_topk_pp"]), p(x["p_t"]), x["alg1_better_in"], x["alg1_worse_in"],
                 f(x["alg1_minus_all_pp"])] for x in out["R6_algorithm1"]["summary"]]
        L.append(table(["θ_CKA", "θ_unique (%)", "pools", "mean subset size", "Alg 1 − top-k (pts)", "p",
                        "better in", "worse in", "Alg 1 − all members"], rows))

    # R7
    if "R7_similarity" in out:
        L += ["## R7. Other similarity measures on the real pools (review points 5, 6; Q4)", "",
              "Candidate-level, within-pool-centred regression of utility on each measure and on accuracy "
              "(cluster-robust by pool); regret of choosing the least similar candidate by that measure; and the "
              "matched pairs re-oriented so that D = utility of the more similar minus the less similar candidate "
              "under that measure. `logit_*` measures use the held-out probe logits, a class-discriminative "
              "representation.", ""]
        for s in SUBS:
            v = out["R7_similarity"].get(s)
            if not v:
                continue
            rows = []
            for m in MEASURES:
                x = v[m]
                md = x.get("matched_D_oriented_by_measure") or {}
                rows.append([m, f(x["coef_pp_per_unit"]), p(x["p"]), f(x["acc_coef"], 3), p(x["acc_p"]),
                             f(x["spearman_with_utility_within_pool"]), f(x["regret_choose_least_similar_pp"], sign=False),
                             f(md.get("estimate")) + " " + ci(md)])
            L.append(f"**{s}** (G = {v[MEASURES[0]]['G']}, candidates = {v[MEASURES[0]]['n']})\n")
            L.append(table(["measure", "coef (pts/unit)", "p", "accuracy coef", "p", "ρ with utility",
                            "regret: least similar", "matched D by measure [95% CI]"], rows))

    # R8
    r8 = out["R8_metrics"]
    L += ["## R8. Other outcome metrics and per-pool intervals (review point 8)", "", r8["sign_note"] + ".", ""]
    rows = []
    for s in SUBS:
        v = r8.get(s) or {}
        for k in ("macro_f1", "nll"):
            x = v.get(k) or {}
            a, pr = x.get("all_pairs") or {}, x.get("primary") or {}
            rows.append([s, k, f(a.get("estimate"), 3), ci(a), f(pr.get("estimate"), 3), ci(pr)])
    L.append(table(["subset", "metric", "D all pairs", "95% CI", "D primary", "95% CI"], rows))
    rows = [[x["pool"], x["domain"], f"{x['high']} vs {x['low']}", f(x["D"]),
             f"[{f(x['fold_ci_low'])}, {f(x['fold_ci_high'])}]"] for x in r8["per_pool_primary"]]
    L.append("Per-pool primary pair with a fold-level interval (5 folds, t(4); folds share training data, so "
             "these intervals are descriptive):\n")
    L.append(table(["pool", "domain", "pair (higher vs lower CKA)", "D", "fold 95% CI"], rows))

    # R9
    L += ["## R9. Matched versus unmatched comparisons across all pools (review point 9)", ""]
    rows = []
    for s in SUBS:
        v = out["R9_unmatched"].get(s)
        if not v:
            continue
        m, u, g = v["matched_all_pairs"] or {}, v["unmatched_all_pairs"] or {}, v["unmatched_D_on_signed_gap"] or {}
        rows.append([s, f(m.get("estimate")) + " " + ci(m), v["unmatched_pairs"], f(u.get("estimate")) + " " + ci(u),
                     f"{v['unmatched_frac_higher_cka_more_accurate']:.0%}", f(g.get("slope_pp_per_point"), 3),
                     p(g.get("p")), f(g.get("intercept")), p(g.get("intercept_p"))])
    L.append(table(["subset", "matched D [95% CI]", "unmatched pairs", "unmatched D [95% CI]",
                    "unmatched: higher-CKA more accurate", "D per signed-gap point", "p",
                    "D at zero gap", "p"], rows))
    return "\n".join(L)


def figures(out, fig_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    os.makedirs(fig_dir, exist_ok=True)
    col = {"pathology": "#b03a2e", "vision": "#1f618d", "text": "#1e8449"}
    paths = []
    # per-pool primary D with fold intervals
    P = out["R8_metrics"]["per_pool_primary"]
    if P:
        fig, ax = plt.subplots(figsize=(6.2, 0.32 * len(P) + 1.2))
        for i, x in enumerate(P[::-1]):
            ax.errorbar(x["D"], i, xerr=[[x["D"] - x["fold_ci_low"]], [x["fold_ci_high"] - x["D"]]],
                        fmt="o", color=col.get(x["domain"], "k"), ms=4, capsize=2, lw=1)
        ax.axvline(0, color="k", lw=0.8)
        ax.axvspan(-0.5, 0.5, color="0.9", zorder=0)
        ax.set_yticks(range(len(P)))
        ax.set_yticklabels([x["pool"] for x in P[::-1]], fontsize=7)
        ax.set_xlabel("D (pp), primary pair, fold-level 95% CI (descriptive)", fontsize=8)
        fig.tight_layout()
        p_ = os.path.join(fig_dir, "per_pool_primary_D_with_fold_CI.png")
        fig.savefig(p_, dpi=200); plt.close(fig); paths.append(p_)
    return paths

## Run (resumable: finished sections are loaded, not recomputed)

In [ ]:
import json, importlib, warnings, torch
import reviewer as RV, reviewer2 as R2, reviewer2_report as RR2
for m in (RV, R2, RR2):
    importlib.reload(m)
warnings.filterwarnings("ignore")
OUT = os.path.join(CFG.RESULTS_DIR, "reviewer")
P = os.path.join(OUT, "reviewer2_analyses.json")
o = json.load(open(P)) if os.path.exists(P) else {"status": R2.EXPLORATORY}
recs = RV.load()
rvj = json.load(open(os.path.join(OUT, "reviewer_analyses.json")))
RV.check_reconstruction(recs)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
steps = [("S1_pilot_overlap_exclusion", lambda: R2.pilot_overlap_exclusion(recs)),
         ("S2_gated_rule", lambda: R2.gated_rule(recs, rvj)),
         ("S5_association_consistency", lambda: R2.association_consistency(rvj)),
         ("S6_multiplicity", lambda: R2.multiplicity(rvj)),
         ("S7_primary_pairs", lambda: R2.primary_pairs(recs)),
         ("S4_c_sensitivity", lambda: R2.c_sensitivity(recs)),
         ("S3_nested_regret", lambda: R2.nested_regret(recs)),
         ("S8_encoder_validation", lambda: R2.encoder_validation(
             __import__("numpy").asarray(__import__("imgcache").ensure_images("dtd")[0][:256]), device=DEV))]
for key, fn in steps:
    if key in o:
        print(f"{key}: already done"); continue
    print(f"{key}: running ...", flush=True)
    o[key] = fn()
    with open(P, "w") as fh:
        json.dump(o, fh, indent=1, default=float)
    print(f"{key}: saved", flush=True)

## Report

In [ ]:
md_text = RR2.build_md(o)
with open(os.path.join(OUT, "reviewer2_analyses.md"), "w") as fh:
    fh.write(md_text)
from IPython.display import Markdown, display
display(Markdown(md_text))
print("written:", os.path.join(OUT, "reviewer2_analyses.md"))